In [9]:
import os
import glob
import re
import json
import pandas as pd

# =========================
# PATH INPUT (CSV)
# =========================
BASE_CSV_DIR = "../Results/CSV"   # sesuaikan kalau beda

# =========================
# OUTPUT PNG (HARUS ADA)
# =========================
IMG_DIR = "../Results/Visualisasi/PerHotel"
os.makedirs(IMG_DIR, exist_ok=True)

# =========================
# OUTPUT DASHBOARD
# =========================
DASHBOARD_DIR = "../Results/Dashboard"
os.makedirs(DASHBOARD_DIR, exist_ok=True)

HTML_PATH = os.path.join(DASHBOARD_DIR, "dashboard_hotel.html")

print("CSV source :", BASE_CSV_DIR)
print("IMG output :", IMG_DIR)
print("HTML output:", HTML_PATH)

CSV source : ../Results/CSV
IMG output : ../Results/Visualisasi/PerHotel
HTML output: ../Results/Dashboard\dashboard_hotel.html


In [10]:
all_files = glob.glob(os.path.join(BASE_CSV_DIR, "**", "*.csv"), recursive=True)

print("Total file CSV ditemukan:", len(all_files))
all_files[:5]

Total file CSV ditemukan: 230


['../Results/CSV\\Analisis_BUMN_All.csv',
 '../Results/CSV\\Analisis_KOMPETITOR_All.csv',
 '../Results/CSV\\Analisis_Master_Lengkap.csv',
 '../Results/CSV\\BUMNB3\\Analisis_Banaran9ResortHotel_Clean.csv',
 '../Results/CSV\\BUMNB3\\Analisis_BrothersSolo_Clean.csv']

In [11]:
dfs = []
for f in all_files:
    try:
        df = pd.read_csv(f)
        df["__source_file"] = os.path.basename(f)
        dfs.append(df)
    except Exception as e:
        print("Gagal baca:", f, "|", e)

df_all = pd.concat(dfs, ignore_index=True)

print("df_all shape:", df_all.shape)
df_all.head()


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4560\2260628720.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


df_all shape: (497844, 13)


,User Name,Review Time,Rating,clean_text,Tipe,Kelas,Nama_Hotel,AI_Sentiment,AI_Aspek,__source_file,AI_Sentiment_Score,AI_Primary_Theme,AI_All_Themes
0,Alifia Lifi,2026.0,5,dulu beberapa tahun yang lalu pernah kesini mu...,BUMN,Bintang 3,Banaran9ResortHotel,positive,Kualitas Makanan & Restoran,Analisis_BUMN_All.csv,NaN,NaN,NaN
1,Desi “Dsy” Rizky,2025.0,3,saya kesini di akhr bulan mei sprtinya masih l...,BUMN,Bintang 3,Banaran9ResortHotel,negative,Kebersihan & Kenyamanan Kamar,Analisis_BUMN_All.csv,NaN,NaN,NaN
2,Test Testing,2025.0,1,Saya beli ayam goreng 3 untuk 40rb tau ngak uk...,BUMN,Bintang 3,Banaran9ResortHotel,negative,Kualitas Makanan & Restoran,Analisis_BUMN_All.csv,NaN,NaN,NaN
3,Fanyf Sy,2025.0,1,rencana buku 4 villa buat gathering keluarga b...,BUMN,Bintang 3,Banaran9ResortHotel,negative,Infrastruktur (AC/WiFi/Parkir/Air),Analisis_BUMN_All.csv,NaN,NaN,NaN
4,Ficha Chandra,2025.0,5,dapat kamar dengan view rawa pening indah bang...,BUMN,Bintang 3,Banaran9ResortHotel,positive,Kebersihan & Kenyamanan Kamar,Analisis_BUMN_All.csv,NaN,NaN,NaN


In [12]:
required_cols = ["Tipe", "Kelas", "Nama_Hotel", "AI_Sentiment", "AI_Aspek"]

missing = [c for c in required_cols if c not in df_all.columns]
if missing:
    raise ValueError(f"Kolom wajib hilang: {missing}")

print("✅ Semua kolom wajib tersedia.")


✅ Semua kolom wajib tersedia.


In [13]:
def safe_filename(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace("/", "-").replace("\\", "-")
    text = re.sub(r"[^a-zA-Z0-9 _\-]", "", text)
    return text

df_hotels = (
    df_all[["Tipe", "Kelas", "Nama_Hotel"]]
    .drop_duplicates()
    .sort_values(["Tipe", "Kelas", "Nama_Hotel"])
    .reset_index(drop=True)
)

df_hotels["safe_name"] = df_hotels["Nama_Hotel"].apply(safe_filename)

df_hotels["pie_file"] = df_hotels.apply(
    lambda r: f"{r['Tipe']}_{r['Kelas']}_{r['safe_name']}_PIE.png",
    axis=1
)

df_hotels["map_file"] = df_hotels.apply(
    lambda r: f"{r['Tipe']}_{r['Kelas']}_{r['safe_name']}_MASALAH.png",
    axis=1
)

hotel_data = df_hotels[["Tipe", "Kelas", "Nama_Hotel", "pie_file", "map_file"]].to_dict("records")

print("Total hotel:", len(hotel_data))
df_hotels.head()


Total hotel: 227


,Tipe,Kelas,Nama_Hotel,safe_name,pie_file,map_file
0,BUMN,Bintang 3,Banaran9ResortHotel,Banaran9ResortHotel,BUMN_Bintang 3_Banaran9ResortHotel_PIE.png,BUMN_Bintang 3_Banaran9ResortHotel_MASALAH.png
1,BUMN,Bintang 3,BrothersSolo,BrothersSolo,BUMN_Bintang 3_BrothersSolo_PIE.png,BUMN_Bintang 3_BrothersSolo_MASALAH.png
2,BUMN,Bintang 3,CodiaBanjarmasin,CodiaBanjarmasin,BUMN_Bintang 3_CodiaBanjarmasin_PIE.png,BUMN_Bintang 3_CodiaBanjarmasin_MASALAH.png
3,BUMN,Bintang 3,CordiaBanjarmasin,CordiaBanjarmasin,BUMN_Bintang 3_CordiaBanjarmasin_PIE.png,BUMN_Bintang 3_CordiaBanjarmasin_MASALAH.png
4,BUMN,Bintang 3,HAKAHotelSemarang,HAKAHotelSemarang,BUMN_Bintang 3_HAKAHotelSemarang_PIE.png,BUMN_Bintang 3_HAKAHotelSemarang_MASALAH.png


In [14]:
# RELATIVE PATH dari Results/Dashboard/dashboard_hotel.html
# menuju Results/Visualisasi/PerHotel/
IMG_REL_DIR = "../Visualisasi/PerHotel"

html_content = f"""
<!DOCTYPE html>
<html lang="id">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Dashboard Analisis Ulasan Hotel</title>

  <style>
    body {{
      font-family: Arial, sans-serif;
      margin: 0;
      background: #f7f7f7;
      color: #222;
    }}

    header {{
      background: #069494;
      color: white;
      padding: 18px 22px;
    }}

    header h1 {{
      margin: 0;
      font-size: 20px;
      font-weight: 700;
    }}

    header p {{
      margin: 6px 0 0;
      opacity: 0.9;
      font-size: 13px;
    }}

    .container {{
      padding: 18px 22px;
    }}

    .filters {{
      display: grid;
      grid-template-columns: 1fr 1fr 2fr;
      gap: 12px;
      margin-bottom: 18px;
    }}

    select, input {{
      padding: 10px 12px;
      border-radius: 10px;
      border: 1px solid #ddd;
      font-size: 14px;
      outline: none;
      background: white;
    }}

    .stats {{
      margin: 10px 0 18px;
      font-size: 13px;
      color: #555;
    }}

    .grid {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(380px, 1fr));
      gap: 16px;
    }}

    .card {{
      background: white;
      border-radius: 14px;
      padding: 14px;
      box-shadow: 0 3px 10px rgba(0,0,0,0.07);
    }}

    .card h3 {{
      margin: 0 0 6px;
      font-size: 15px;
      line-height: 1.3;
    }}

    .meta {{
      font-size: 12px;
      color: #666;
      margin-bottom: 10px;
    }}

    .img-wrap {{
      display: grid;
      grid-template-columns: 1fr;
      gap: 12px;
    }}

    .img-wrap img {{
      width: 100%;
      border-radius: 12px;
      border: 1px solid #eee;
      background: #fafafa;
    }}

    .badge {{
      display: inline-block;
      padding: 4px 10px;
      border-radius: 999px;
      font-size: 11px;
      margin-right: 6px;
      background: #eee;
      color: #333;
    }}

    .bumn {{ background: #e8f7f7; color: #069494; }}
    .kompetitor {{ background: #fff1e8; color: #B7410E; }}

    footer {{
      padding: 18px 22px;
      font-size: 12px;
      color: #777;
      text-align: center;
    }}
  </style>
</head>

<body>
  <header>
    <h1>Dashboard Analisis Ulasan Hotel</h1>
    <p>Pie Chart Sentimen + Peta Masalah (Top 6) untuk setiap hotel</p>
  </header>

  <div class="container">
    <div class="filters">
      <select id="filterTipe">
        <option value="ALL">Semua Tipe</option>
        <option value="BUMN">BUMN</option>
        <option value="KOMPETITOR">KOMPETITOR</option>
      </select>

      <select id="filterKelas">
        <option value="ALL">Semua Bintang</option>
        <option value="Bintang3">Bintang 3</option>
        <option value="Bintang4">Bintang 4</option>
        <option value="Bintang5">Bintang 5</option>
      </select>

      <input id="searchHotel" type="text" placeholder="Cari nama hotel..." />
    </div>

    <div class="stats" id="stats"></div>

    <div class="grid" id="hotelGrid"></div>
  </div>

  <footer>
    Generated otomatis dari hasil analisis ulasan hotel.
  </footer>

<script>
const IMG_DIR = "{IMG_REL_DIR}";
const hotelData = {json.dumps(hotel_data, ensure_ascii=False)};

const filterTipe = document.getElementById("filterTipe");
const filterKelas = document.getElementById("filterKelas");
const searchHotel = document.getElementById("searchHotel");
const hotelGrid = document.getElementById("hotelGrid");
const stats = document.getElementById("stats");

function render() {{
  const tipeVal = filterTipe.value;
  const kelasVal = filterKelas.value;
  const q = searchHotel.value.toLowerCase().trim();

  const filtered = hotelData.filter(h => {{
    const okTipe = (tipeVal === "ALL") || (h.Tipe === tipeVal);
    const okKelas = (kelasVal === "ALL") || (h.Kelas === kelasVal);
    const okSearch = (q === "") || (h.Nama_Hotel.toLowerCase().includes(q));
    return okTipe && okKelas && okSearch;
  }});

  stats.innerHTML = `Menampilkan <b>${{filtered.length}}</b> dari <b>${{hotelData.length}}</b> hotel`;

  hotelGrid.innerHTML = "";

  filtered.forEach(h => {{
    const badgeClass = (h.Tipe === "BUMN") ? "bumn" : "kompetitor";

    const card = document.createElement("div");
    card.className = "card";

    card.innerHTML = `
      <h3>${{h.Nama_Hotel}}</h3>
      <div class="meta">
        <span class="badge ${{badgeClass}}">${{h.Tipe}}</span>
        <span class="badge">${{h.Kelas.replace("Bintang", "Bintang ")}}</span>
      </div>

      <div class="img-wrap">
        <img src="${{IMG_DIR}}/${{h.pie_file}}" alt="Pie Chart Sentimen" />
        <img src="${{IMG_DIR}}/${{h.map_file}}" alt="Peta Masalah Negatif" />
      </div>
    `;

    hotelGrid.appendChild(card);
  }});
}}

filterTipe.addEventListener("change", render);
filterKelas.addEventListener("change", render);
searchHotel.addEventListener("input", render);

render();
</script>

</body>
</html>
"""

with open(HTML_PATH, "w", encoding="utf-8") as f:
    f.write(html_content)

print("✅ Dashboard HTML berhasil dibuat:", HTML_PATH)


✅ Dashboard HTML berhasil dibuat: ../Results/Dashboard\dashboard_hotel.html


In [15]:
missing_pie = 0
missing_map = 0

for _, r in df_hotels.iterrows():
    pie_path = os.path.join(IMG_DIR, r["pie_file"])
    map_path = os.path.join(IMG_DIR, r["map_file"])

    if not os.path.exists(pie_path):
        missing_pie += 1
    if not os.path.exists(map_path):
        missing_map += 1

print("Missing PIE :", missing_pie)
print("Missing MAP :", missing_map)


Missing PIE : 227
Missing MAP : 227


In [8]:
import os
import glob

IMG_DIR = "../Results/Visualisasi/PerHotel"

files = glob.glob(os.path.join(IMG_DIR, "*.png"))
print("Total PNG:", len(files))
files[:10]


Total PNG: 454


['../Results/Visualisasi/PerHotel\\BUMN_Bintang3_Banaran9ResortHotel_MASALAH.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_Banaran9ResortHotel_PIE.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_BrothersSolo_MASALAH.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_BrothersSolo_PIE.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_CodiaBanjarmasin_MASALAH.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_CodiaBanjarmasin_PIE.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_CordiaBanjarmasin_MASALAH.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_CordiaBanjarmasin_PIE.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_HAKAHotelSemarang_MASALAH.png',
 '../Results/Visualisasi/PerHotel\\BUMN_Bintang3_HAKAHotelSemarang_PIE.png']